# S4.1 — Classification A/B after zero separation

**Purpose:** Run the same classifier twice per panel, so zeros are not mixed into the official continuous labels.

1. **A — all scatter points** — every finite land cell, including exact zeros and near-zeros. Writes `output/S4.1/classification_all_points.csv`.
2. **B — drop 0 and near-0** — keep only `|x| > 1e-4` AND `|y| > 1e-4`. Writes `output/S4.1/classification_continuous.csv`. This is the table **S4.2** reads.

The early table below is an **exact-zero inventory** (`v == 0` only). It is **not** the classification A vs B. A vs B comes later and reports how many panels lost points vs how many labels actually changed.

**Figures** (class-coloured scatter + hexbin, LOWESS) live in `S4.2_CMIP6_continuous_scatter_fits.ipynb`. S4.2 uses **B**.

**Classifier:** `classifier0825-branch.py` (CSV rows store `classifier_sha256`)

**Data:** RAW only, no trimming.

Near-zero here means `0 < |v| ≤ 1e-4` (tiny numerical + effective zero). Exact zero is `v = 0`.


In [1]:
from __future__ import annotations

import importlib.util
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from IPython.display import Image, display

warnings.filterwarnings('ignore', category=FutureWarning)

plt.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 180,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'font.family': 'DejaVu Sans', 'axes.edgecolor': '#30343B',
    'axes.linewidth': 0.8, 'axes.titlesize': 9,
    'axes.titleweight': 'semibold', 'axes.labelsize': 8,
    'xtick.labelsize': 7, 'ytick.labelsize': 7,
})


def locate_case_dir() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / 'cmip_utils.py').exists() and candidate.name == 'caseA':
            return candidate
        nested = candidate / 'case' / 'caseA'
        if (nested / 'cmip_utils.py').exists():
            return nested
    raise FileNotFoundError('Could not locate case/caseA/')


CASE_DIR = locate_case_dir()
DATA_ROOT = Path('/Volumes/mimi-T9/CMIP6')

MODELS = ['CESM2', 'CNRM-CM6-1', 'CanESM5', 'GFDL-CM4', 'CMCC-CM2-SR5']
ZONES = ['all_land', 'WW', 'WD', 'CW', 'CD', 'LI']
ZONE_SHORT = {
    'all_land': 'All land (all_land)',
    'WW': 'Wet-warm (WW)',
    'WD': 'Dry-warm (WD)',
    'CW': 'Wet-cold (CW)',
    'CD': 'Dry-cold (CD)',
    'LI': 'Land ice (LI)',
}

VARIABLE_PAIRS = [
    ('P',          'Q',  'P → Q'),
    ('ET',         'Q',  'ET → Q'),
    ('mrros',      'Q',  'mrros → Q'),
    ('prsn',       'Q',  'prsn → Q'),
    ('tran',       'Q',  'tran → Q'),
    ('evspsblsoi', 'Q',  'evspsblsoi → Q'),
    ('hfls',       'Q',  'hfls → Q'),
    ('hfss',       'Q',  'hfss → Q'),
    ('lai',        'Q',  'lai → Q'),
    ('tas',        'Q',  'tas → Q'),
    ('rsds',       'Q',  'rsds → Q'),
    ('mrso',       'Q',  'mrso → Q'),
    ('mrsos',      'Q',  'mrsos → Q'),
    ('rlds',       'Q',  'rlds → Q'),
    ('rlus',       'Q',  'rlus → Q'),
    ('rsus',       'Q',  'rsus → Q'),
    ('P',          'ET', 'P → ET')
]

START_YEAR, END_YEAR = 1985, 2014

# Classifier
CLASSIFIER_PATH = CASE_DIR / 'classifier0825-branch.py'
assert CLASSIFIER_PATH.exists(), f'Classifier not found: {CLASSIFIER_PATH}'

OUTPUT_DIR = CASE_DIR / 'output' / 'S4.1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Case directory:  {CASE_DIR}')
print(f'Classifier:      {CLASSIFIER_PATH.name}')
print(f'Output:          {OUTPUT_DIR}')
print(f'Variable pairs:  {len(VARIABLE_PAIRS)}')
print('Figures:          S4.2_CMIP6_continuous_scatter_fits.ipynb (reads classification CSV)')


Case directory:  /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA
Classifier:      classifier0825-branch.py
Output:          /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4.1
Variable pairs:  17
Figures:          S4.2_CMIP6_continuous_scatter_fits.ipynb (reads classification CSV)


In [2]:
# --- Load zone climatology for all models ---
runs = []
for model in MODELS:
    search_roots = [
        DATA_ROOT / model / 'historical',
        CASE_DIR / 'data' / model / 'historical',
    ]
    zone_name = f'zone_climatology_{START_YEAR}_{END_YEAR}.parquet'
    found = False
    for sr in search_roots:
        if not sr.exists():
            continue
        zone_paths = sorted(sr.glob(f'*/*/land/zones/{zone_name}'))
        if zone_paths:
            t = pd.read_parquet(zone_paths[0])
            t.rename(columns={'R': 'Q'}, inplace=True)
            runs.append({'model': model, 'data': t})
            print(f'{model}: {len(t)} rows')
            found = True
            break
    if not found:
        print(f'{model}: SKIPPED')

print(f'\nLoaded: {len(runs)} models')

CESM2: 21013 rows
CNRM-CM6-1: 13639 rows
CanESM5: 3464 rows
GFDL-CM4: 22514 rows
CMCC-CM2-SR5: 20675 rows

Loaded: 5 models


In [3]:
# --- Helpers: data extraction, zero grouping, S4-style drawing utilities ---

from matplotlib.colors import LogNorm, to_rgba
import matplotlib.patches as mpatches


def get_xy(run, x_var, y_var, zone):
    """Return (x, y) RAW finite arrays for a model-run × zone."""
    data = run['data']
    if x_var not in data.columns or y_var not in data.columns:
        return None, None
    if zone == 'all_land':
        sub = data
    else:
        sub = data[data['analysis_zone'] == zone]
    if len(sub) < 5:
        return None, None
    x = sub[x_var].values.astype(float)
    y = sub[y_var].values.astype(float)
    finite = np.isfinite(x) & np.isfinite(y)
    return x[finite], y[finite]


def zero_groups(x, y):
    """Split into four quadrant masks."""
    xz = x == 0.0
    yz = y == 0.0
    return {
        'both_zero': xz & yz,
        'x_zero':    xz & ~yz,
        'y_zero':    ~xz & yz,
        'cont':      ~xz & ~yz,
    }


# ---------- S4-style visual utilities (font sizes match S4) ----------

UNIT_LABELS = {
    'P': 'Precipitation, P (mm yr⁻¹)',
    'ET': 'Evapotranspiration, ET (mm yr⁻¹)',
    'Q': 'Total runoff, Q (mm yr⁻¹)',
    'hfls': 'Latent heat flux (W m⁻²)',
    'hfss': 'Sensible heat flux (W m⁻²)',
    'tran': 'Transpiration (mm yr⁻¹)',
    'evspsblsoi': 'Soil evaporation (mm yr⁻¹)',
    'mrros': 'Surface runoff (mm yr⁻¹)',
    'mrso': 'Total soil moisture (kg m⁻²)',
    'mrsos': 'Topsoil moisture (kg m⁻²)',
    'lai': 'Leaf area index (m² m⁻²)',
    'tas': 'Near-surface temperature (K)',
    'prsn': 'Snowfall (mm yr⁻¹)',
    'rlds': 'Downward LW radiation (W m⁻²)',
    'rlus': 'Upward LW radiation (W m⁻²)',
    'rsds': 'Downward SW radiation (W m⁻²)',
    'rsus': 'Upward SW radiation (W m⁻²)',
}

VAR_FULL_NAMES = {}
for _var, _label in UNIT_LABELS.items():
    _name = _label.split('(')[0].strip()
    if ', ' in _name:
        _name = _name.rsplit(', ', 1)[0]
    VAR_FULL_NAMES[_var] = _name


def full_pair_title(x_var, y_var):
    return f'{VAR_FULL_NAMES[x_var]} ({x_var}) → {VAR_FULL_NAMES[y_var]} ({y_var})'


def style_axes(ax):
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('#30343B')
        spine.set_linewidth(1.0)
    ax.tick_params(
        direction='out', length=4.0, width=0.9,
        color='#30343B', labelsize=11,
    )
    ax.grid(True, color='#D3D3D3', linewidth=0.6, alpha=0.7, zorder=0)


def padded_limits(values, pad_fraction=0.04):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return (-1.0, 1.0)
    lower, upper = float(values.min()), float(values.max())
    span = upper - lower
    if not np.isfinite(span) or span == 0:
        span = max(abs(lower), 1.0)
    pad = pad_fraction * span
    return lower - pad, upper + pad


COL_W, ROW_H = 3.2, 2.8


def draw_density_hexbin(ax, x_values, y_values, cmap='Reds',
                       gridsize=26, extent=None):
    """Hexbin density, S5-style: integer gridsize, optional shared extent."""
    x_arr = np.asarray(x_values, dtype=float)
    y_arr = np.asarray(y_values, dtype=float)
    finite = np.isfinite(x_arr) & np.isfinite(y_arr)
    xf, yf = x_arr[finite], y_arr[finite]
    if len(xf) < 20 or np.ptp(xf) <= 0 or np.ptp(yf) <= 0:
        ax.scatter(xf, yf, s=5, c='k', alpha=0.4, rasterized=True)
        return
    kw = dict(
        gridsize=gridsize, cmap=cmap, mincnt=1,
        norm=LogNorm(), linewidths=0.0, zorder=1,
    )
    if extent is not None:
        kw['extent'] = extent
    ax.hexbin(xf, yf, **kw)


def column_limits_for_pair(x_var, y_var, zone, runs_list):
    """Shared axis limits across all models for a given zone."""
    xs, ys = [], []
    for run in runs_list:
        x, y = get_xy(run, x_var, y_var, zone)
        if x is not None and len(x) > 0:
            xs.append(x)
            ys.append(y)
    if not xs:
        return (-1.0, 1.0), (-1.0, 1.0)
    return padded_limits(np.concatenate(xs)), padded_limits(np.concatenate(ys))


# ---------- Zero-point colours ----------

ZERO_COLORS = {
    'cont':      '#444444',
    'x_zero':    '#C02020',
    'y_zero':    '#2050B0',
    'both_zero': '#7020A0',
}
ZERO_SIZES = {
    'cont':      1.0,
    'x_zero':    3.0,
    'y_zero':    3.0,
    'both_zero': 4.0,
}
ZERO_ALPHAS = {
    'cont':      0.5,
    'x_zero':    0.8,
    'y_zero':    0.8,
    'both_zero': 0.9,
}
ZERO_LABELS = {
    'cont':      'Continuous (x≠0, y≠0)',
    'x_zero':    'x=0, y≠0',
    'y_zero':    'x≠0, y=0',
    'both_zero': 'x=0, y=0',
}

CLASS_FACECOLORS = {
    'Linear': '#FFE8D2',
    'Near-linear': '#F5F0E6',
    'Acceleration': '#FFF6CC',
    'Saturation': '#F8E4D8',
    'Branch': '#DDEAF6',
    'Candidate branch': '#E6DEF4',
    'Complex': '#DFF2F2',
    'No simple': '#F0F0F0',
    'Uncertain': '#F0F0F0',
    'No Global Relationship': '#FFFFFF',
    'Skipped (too few cont)': '#FFFFFF',
}
CLASS_FACE_ALPHA = {
    'Linear': 0.40,
    'Near-linear': 0.40,
    'Acceleration': 0.40,
    'Saturation': 0.40,
    'Branch': 0.35,
    'Candidate branch': 0.35,
    'Complex': 0.35,
    'No simple': 0.30,
    'Uncertain': 0.30,
}
CLASS_TEXTCOLORS = {
    'Linear': '#C45C26',
    'Near-linear': '#7A6A48',
    'Acceleration': '#B88600',
    'Saturation': '#C45C3A',
    'Branch': '#2F6AA0',
    'Candidate branch': '#6B4C9A',
    'Complex': '#2A7A7A',
    'No simple': '#666666',
    'Uncertain': '#666666',
    'No Global Relationship': '#888888',
    'Skipped (too few cont)': '#888888',
}


def class_face(group):
    """Light, transparent panel background by classification."""
    if group is None or (isinstance(group, float) and np.isnan(group)):
        return to_rgba('#FFFFFF', 1.0)
    key = str(group)
    return to_rgba(
        CLASS_FACECOLORS.get(key, '#FFFFFF'),
        CLASS_FACE_ALPHA.get(key, 0.25),
    )


def class_text_color(group):
    if group is None or (isinstance(group, float) and np.isnan(group)):
        return '#333333'
    return CLASS_TEXTCOLORS.get(str(group), '#333333')


def draw_panel_class_label(ax, group, detail_lines, fontsize=8.4):
    """Bold class name in category colour; metrics in dark grey."""
    from matplotlib.offsetbox import AnnotationBbox, TextArea, VPacker
    children = []
    label = '' if group is None else str(group)
    if label and label != 'nan':
        children.append(TextArea(
            label,
            textprops={
                'fontsize': fontsize + 0.6,
                'fontweight': 'bold',
                'color': class_text_color(group),
                'family': 'DejaVu Sans',
            },
        ))
    if detail_lines:
        if isinstance(detail_lines, (list, tuple)):
            detail = '\n'.join(str(x) for x in detail_lines if x)
        else:
            detail = str(detail_lines)
        if detail:
            children.append(TextArea(
                detail,
                textprops={
                    'fontsize': fontsize,
                    'fontweight': 'normal',
                    'color': '#202020',
                    'family': 'DejaVu Sans',
                },
            ))
    if not children:
        return
    pack = VPacker(children=children, align='left', pad=0, sep=2)
    edge = class_text_color(group) if label and label != 'nan' else '#C9C9C9'
    box = AnnotationBbox(
        pack, (0.03, 0.97), xycoords='axes fraction',
        box_alignment=(0.0, 1.0), frameon=True, pad=0.28,
        bboxprops={
            'boxstyle': 'round,pad=0.30',
            'facecolor': 'white',
            'edgecolor': edge,
            'linewidth': 0.9,
            'alpha': 0.92,
        },
        zorder=8,
    )
    ax.add_artist(box)


def canonical_group(group, path_trace=''):
    """Map old SiZer Uncertain/Complex labels onto No simple / Complex."""
    g = '' if group is None else str(group)
    pt = '' if path_trace is None else str(path_trace)
    simple_try = ('Pearson:Simple' in pt) or ('Simple(mid)' in pt)
    if g in {'Uncertain', 'Complex'} and simple_try:
        return 'No simple'
    if g == 'Uncertain':
        return 'Complex'
    return g


def classification_lookup():
    """(Pair, Model, Zone) → group from df_clf or the saved CSV."""
    lookup = {}
    src = globals().get('df_clf')
    if not isinstance(src, pd.DataFrame) or 'group' not in getattr(src, 'columns', []):
        path = OUTPUT_DIR / 'classification_continuous.csv'
        if not path.exists():
            return lookup
        src = pd.read_csv(path)
    for rec in src.itertuples(index=False):
        lookup[(rec.Pair, rec.Model, rec.Zone)] = canonical_group(
            rec.group, getattr(rec, 'path_trace', ''),
        )
    return lookup

print('Helpers defined (S4-style).')
print('Example title:', full_pair_title('prsn', 'Q'))

Helpers defined (S4-style).
Example title: Snowfall (prsn) → Total runoff (Q)


In [4]:
# --- Exact-zero inventory (v == 0 only). Not classification A vs B. ---

MIN_CONT = 30
records = []

for run in runs:
    model = run['model']
    for x_var, y_var, pair_label in VARIABLE_PAIRS:
        for zone in ZONES:
            x, y = get_xy(run, x_var, y_var, zone)
            if x is None or len(x) < 5:
                continue
            g = zero_groups(x, y)
            n_all = len(x)
            n_cont = int(g['cont'].sum())
            n_xz = int(g['x_zero'].sum())
            n_yz = int(g['y_zero'].sum())
            n_bz = int(g['both_zero'].sum())
            n_zero = n_xz + n_yz + n_bz
            ret = n_cont / n_all if n_all > 0 else 0

            if n_zero == 0:
                status = 'No exact zero'
            elif n_cont < MIN_CONT:
                status = 'Too few non-zero'
            else:
                status = 'Has exact zero'

            records.append({
                'Pair': pair_label,
                'Model': model,
                'Zone': zone,
                'N_all': n_all,
                'p00': n_bz,
                'p01_x0': n_xz,
                'p10_y0': n_yz,
                'N_nonzero': n_cont,
                'Retention': f'{100 * ret:.1f}%',
                'Status': status,
            })

df_exact_zero = pd.DataFrame(records)
status_order = ['No exact zero', 'Has exact zero', 'Too few non-zero']

print('=== Exact-zero inventory (v == 0 only) ===')
print('Not classification A vs B. A vs B uses |v| > 1e-4 and is later in this notebook.\n')
status_counts = df_exact_zero['Status'].value_counts()
for s in status_order:
    print(f'  {s}: {status_counts.get(s, 0)}')
print(f'  Total panels: {len(df_exact_zero)}\n')

has_zero = df_exact_zero[df_exact_zero['Status'] != 'No exact zero'].copy()
has_zero = has_zero.sort_values(['Pair', 'Model', 'Zone'])
print(f'Panels with at least one exact zero: {len(has_zero)}\n')
with pd.option_context('display.max_rows', 300, 'display.max_columns', 12, 'display.width', 140):
    display(has_zero.reset_index(drop=True))

print('\n=== Per-pair summary ===')
pair_summary = (
    df_exact_zero.groupby('Pair')['Status']
    .value_counts()
    .unstack(fill_value=0)
    .reindex(columns=status_order, fill_value=0)
)
display(pair_summary)

print('\n=== Zone breakdown (panels with exact zeros) ===')
zone_summary = (
    has_zero.groupby(['Zone', 'Status']).size()
    .unstack(fill_value=0)
    .reindex(columns=['Has exact zero', 'Too few non-zero'], fill_value=0)
)
display(zone_summary)

print('\n=== Model breakdown (panels with exact zeros) ===')
model_summary = (
    has_zero.groupby(['Model', 'Status']).size()
    .unstack(fill_value=0)
    .reindex(columns=['Has exact zero', 'Too few non-zero'], fill_value=0)
)
display(model_summary)


=== A/B Impact Summary ===

  Unchanged: 350
  May change: 158
  Zero-dominated: 2
  Total panels: 510

Panels affected by zero separation: 160



,Pair,Model,Zone,N_all,p00,p01_x0,p10_y0,N_cont,Retention,Status
0,ET → Q,CanESM5,WD,710,0,0,21,689,97.0%,May change
1,ET → Q,CanESM5,all_land,3464,0,0,21,3443,99.4%,May change
2,ET → Q,GFDL-CM4,CD,2272,0,0,1,2271,100.0%,May change
3,ET → Q,GFDL-CM4,LI,7273,0,0,6127,1146,15.8%,May change
4,ET → Q,GFDL-CM4,WD,4162,0,0,9,4153,99.8%,May change
5,ET → Q,GFDL-CM4,all_land,22514,0,0,6142,16372,72.7%,May change
6,P → Q,CanESM5,WD,710,0,0,21,689,97.0%,May change
7,P → Q,CanESM5,all_land,3464,0,0,21,3443,99.4%,May change
8,P → Q,GFDL-CM4,CD,2272,0,0,1,2271,100.0%,May change
9,P → Q,GFDL-CM4,LI,7273,0,0,6127,1146,15.8%,May change



=== Per-pair summary ===


Status,Unchanged,May change,Zero-dominated
Pair,,,
ET → Q,24,6,0
P → ET,30,0,0
P → Q,24,6,0
evspsblsoi → Q,20,10,0
hfls → Q,24,6,0
hfss → Q,24,6,0
lai → Q,7,22,1
mrros → Q,12,18,0
mrso → Q,19,11,0



=== Zone breakdown ===


Status,May change,Zero-dominated
Zone,,
CD,23,0
CW,12,0
LI,31,2
WD,41,0
WW,7,0
all_land,44,0



=== Model breakdown ===


Status,May change,Zero-dominated
Model,,
CESM2,21,0
CMCC-CM2-SR5,18,0
CNRM-CM6-1,14,0
CanESM5,39,2
GFDL-CM4,66,0


In [5]:
%%script --no-raise-error false
# Disabled: exact-zero (x==0) 5×6 scatter/hexbin. Use the near-zero 5×6 section instead.
# --- Generate figures for every variable pair (S4 style) ---
# Per pair: 2 figures (5 rows × 6 cols = models × zones)
#   1. Scatter — full RAW, zero points colour-coded
#   2. Hexbin  — Continuous only (x≠0, y≠0), LogNorm, Greys

COL_W, ROW_H = 3.2, 2.8

fig_count = 0

for x_var, y_var, pair_label in VARIABLE_PAIRS:
    pair_slug = f'{x_var}_{y_var}'
    x_label = UNIT_LABELS.get(x_var, x_var)
    y_label = UNIT_LABELS.get(y_var, y_var)
    pair_full = full_pair_title(x_var, y_var)

    # Pre-compute shared column limits per zone (across all models)
    col_limits = {}
    for zone in ZONES:
        col_limits[zone] = column_limits_for_pair(x_var, y_var, zone, runs)
    cls_lookup = classification_lookup()

    n_rows, n_cols = len(MODELS), len(ZONES)

    # ============================================================
    # Figure 1: Scatter — full RAW, zeros colour-coded
    # ============================================================
    fig_sc, axes_sc = plt.subplots(
        n_rows, n_cols,
        figsize=(COL_W * n_cols, ROW_H * n_rows),
        constrained_layout=True, squeeze=False,
    )
    fig_sc.suptitle(
        f'Scatter (RAW) — {pair_full}',
        fontsize=17, fontweight='semibold',
    )

    for row_i, run in enumerate(runs):
        model = run['model']
        for col_j, zone in enumerate(ZONES):
            ax = axes_sc[row_i, col_j]
            ax.set_aspect('auto')
            x, y = get_xy(run, x_var, y_var, zone)
            xlim, ylim = col_limits[zone]
            grp = cls_lookup.get((pair_label, model, zone))
            ax.set_facecolor(class_face(grp))

            if x is None or len(x) < 5:
                ax.text(0.5, 0.5, 'no data', ha='center', va='center',
                        transform=ax.transAxes, fontsize=10, color='#999')
                ax.set_xlim(xlim); ax.set_ylim(ylim)
                style_axes(ax)
                if row_i == 0:
                    ax.set_title(ZONE_SHORT[zone], fontsize=13, fontweight='semibold', pad=6)
                if col_j == 0:
                    ax.set_ylabel(y_label, fontsize=11)
                    ax.text(-0.35, 0.5, model, transform=ax.transAxes,
                            fontsize=12, fontweight='bold', color='#222222',
                            ha='right', va='center', rotation=90)
                if row_i == n_rows - 1:
                    ax.set_xlabel(x_label, fontsize=11)
                continue

            g = zero_groups(x, y)
            n_all = len(x)
            n_cont = int(g['cont'].sum())
            ret = 100 * n_cont / n_all if n_all > 0 else 0

            for key in ['cont', 'x_zero', 'y_zero', 'both_zero']:
                m = g[key]
                if m.sum() == 0:
                    continue
                ax.scatter(x[m], y[m], c=ZERO_COLORS[key],
                           s=ZERO_SIZES[key], alpha=ZERO_ALPHAS[key],
                           linewidths=0, rasterized=True, zorder=2)

            n_xz = int(g['x_zero'].sum())
            n_yz = int(g['y_zero'].sum())
            n_bz = int(g['both_zero'].sum())
            ann_lines = [f'N={n_all}  cont={n_cont} ({ret:.0f}%)']
            parts = []
            if n_xz > 0: parts.append(f'x0={n_xz}')
            if n_yz > 0: parts.append(f'y0={n_yz}')
            if n_bz > 0: parts.append(f'xy0={n_bz}')
            if parts:
                ann_lines.append('  '.join(parts))
            draw_panel_class_label(ax, grp, ann_lines, fontsize=8.4)

            ax.set_xlim(xlim); ax.set_ylim(ylim)
            style_axes(ax)

            if row_i == 0:
                ax.set_title(ZONE_SHORT[zone], fontsize=13, fontweight='semibold', pad=6)
            if col_j == 0:
                ax.set_ylabel(y_label, fontsize=11)
                ax.text(-0.35, 0.5, model, transform=ax.transAxes,
                        fontsize=12, fontweight='bold', color='#222222',
                        ha='right', va='center', rotation=90)
            if row_i == n_rows - 1:
                ax.set_xlabel(x_label, fontsize=11)

    handles = [
        mpatches.Patch(color=ZERO_COLORS[k], label=ZERO_LABELS[k])
        for k in ['cont', 'x_zero', 'y_zero', 'both_zero']
    ]
    fig_sc.legend(handles=handles, loc='lower center', ncol=4,
                  fontsize=11, frameon=True, fancybox=True,
                  framealpha=0.5, edgecolor='none',
                  bbox_to_anchor=(0.5, -0.04))

    sc_path = OUTPUT_DIR / f'scatter_{pair_slug}.png'
    fig_sc.savefig(sc_path, dpi=160, bbox_inches='tight')
    print(f'Saved: {sc_path.name}')
    plt.show()
    fig_count += 1

    # ============================================================
    # Figure 2: Hexbin — Continuous only (x≠0, y≠0)
    # ============================================================
    fig_hc, axes_hc = plt.subplots(
        n_rows, n_cols,
        figsize=(COL_W * n_cols, ROW_H * n_rows),
        constrained_layout=True, squeeze=False,
    )
    fig_hc.suptitle(
        f'Hexbin Density — Continuous only (x≠0 ∧ y≠0) — {pair_full}',
        fontsize=17, fontweight='semibold',
    )

    for row_i, run in enumerate(runs):
        model = run['model']
        for col_j, zone in enumerate(ZONES):
            ax = axes_hc[row_i, col_j]
            ax.set_aspect('auto')
            x, y = get_xy(run, x_var, y_var, zone)
            xlim, ylim = col_limits[zone]
            grp = cls_lookup.get((pair_label, model, zone))
            ax.set_facecolor(class_face(grp))

            if x is None or len(x) < 5:
                ax.text(0.5, 0.5, 'no data', ha='center', va='center',
                        transform=ax.transAxes, fontsize=10, color='#999')
                ax.set_xlim(xlim); ax.set_ylim(ylim)
                style_axes(ax)
                if row_i == 0:
                    ax.set_title(ZONE_SHORT[zone], fontsize=13, fontweight='semibold', pad=6)
                if col_j == 0:
                    ax.set_ylabel(y_label, fontsize=11)
                    ax.text(-0.35, 0.5, model, transform=ax.transAxes,
                            fontsize=12, fontweight='bold', color='#222222',
                            ha='right', va='center', rotation=90)
                if row_i == n_rows - 1:
                    ax.set_xlabel(x_label, fontsize=11)
                continue

            g = zero_groups(x, y)
            n_all = len(x)
            n_cont = int(g['cont'].sum())
            ret = 100 * n_cont / n_all if n_all > 0 else 0
            xc, yc = x[g['cont']], y[g['cont']]

            if n_cont < 20:
                ax.text(0.5, 0.5,
                        f'Zero-dominated\nN_cont={n_cont}',
                        ha='center', va='center',
                        transform=ax.transAxes, fontsize=10, color='#C03030')
                ax.set_xlim(xlim); ax.set_ylim(ylim)
                style_axes(ax)
                if row_i == 0:
                    ax.set_title(ZONE_SHORT[zone], fontsize=13, fontweight='semibold', pad=6)
                if col_j == 0:
                    ax.set_ylabel(y_label, fontsize=11)
                    ax.text(-0.35, 0.5, model, transform=ax.transAxes,
                            fontsize=12, fontweight='bold', color='#222222',
                            ha='right', va='center', rotation=90)
                if row_i == n_rows - 1:
                    ax.set_xlabel(x_label, fontsize=11)
                continue

            draw_density_hexbin(
                ax, xc, yc, cmap='Greys', gridsize=26,
                extent=(xlim[0], xlim[1], ylim[0], ylim[1]),
            )

            draw_panel_class_label(
                ax, grp,
                [f'N_cont={n_cont} ({ret:.0f}%)', f'N_all={n_all}'],
                fontsize=8.4,
            )

            ax.set_xlim(xlim); ax.set_ylim(ylim)
            style_axes(ax)

            if row_i == 0:
                ax.set_title(ZONE_SHORT[zone], fontsize=13, fontweight='semibold', pad=6)
            if col_j == 0:
                ax.set_ylabel(y_label, fontsize=11)
                ax.text(-0.35, 0.5, model, transform=ax.transAxes,
                        fontsize=12, fontweight='bold', color='#222222',
                        ha='right', va='center', rotation=90)
            if row_i == n_rows - 1:
                ax.set_xlabel(x_label, fontsize=11)

    hc_path = OUTPUT_DIR / f'hexbin_cont_{pair_slug}.png'
    fig_hc.savefig(hc_path, dpi=160, bbox_inches='tight')
    print(f'Saved: {hc_path.name}')
    plt.show()
    fig_count += 1

print(f'\nDone. {fig_count} figures saved to {OUTPUT_DIR}')


## Distribution Diagnostic — per-variable marginal histograms

For each unique variable, a 5×6 (models × zones) panel figure.
Each panel shows:
- Histogram with log-scaled y-axis (count) to reveal near-zero structure
- Red dashed line at x=0
- Annotation: N, exact-zero count, min non-zero |value|, P1, P5

In [6]:
# # --- Distribution diagnostic: per-variable 5×6 histogram (log y-axis) ---

# DIAG_DIR = OUTPUT_DIR / 'diag_dist'
# DIAG_DIR.mkdir(parents=True, exist_ok=True)

# ALL_VARS = list(dict.fromkeys(
#     [v for trip in VARIABLE_PAIRS for v in (trip[0], trip[1])]
# ))

# def get_var_values(run, var, zone):
#     data = run['data']
#     if var not in data.columns:
#         return None
#     if zone == 'all_land':
#         sub = data
#     else:
#         sub = data[data['analysis_zone'] == zone]
#     if len(sub) < 5:
#         return None
#     v = sub[var].values.astype(float)
#     return v[np.isfinite(v)]

# COL_W, ROW_H = 3.2, 2.8
# n_rows, n_cols = len(MODELS), len(ZONES)

# for var in ALL_VARS:
#     var_label = UNIT_LABELS.get(var, var)
#     var_full = VAR_FULL_NAMES.get(var, var)

#     fig, axes = plt.subplots(
#         n_rows, n_cols,
#         figsize=(COL_W * n_cols, ROW_H * n_rows),
#         constrained_layout=True, squeeze=False,
#     )
#     fig.suptitle(
#         f'Distribution — {var_full} ({var})',
#         fontsize=17, fontweight='semibold',
#     )

#     for row_i, run in enumerate(runs):
#         model = run['model']
#         for col_j, zone in enumerate(ZONES):
#             ax = axes[row_i, col_j]
#             vals = get_var_values(run, var, zone)

#             if vals is None or len(vals) < 5:
#                 ax.text(0.5, 0.5, 'no data', ha='center', va='center',
#                         transform=ax.transAxes, fontsize=10, color='#999')
#                 style_axes(ax)
#                 if row_i == 0:
#                     ax.set_title(ZONE_SHORT[zone], fontsize=13, fontweight='semibold', pad=6)
#                 if col_j == 0:
#                     ax.set_ylabel('Count', fontsize=11)
#                     ax.text(-0.35, 0.5, model, transform=ax.transAxes,
#                             fontsize=12, fontweight='bold', color='#222222',
#                             ha='right', va='center', rotation=90)
#                 if row_i == n_rows - 1:
#                     ax.set_xlabel(var_label, fontsize=11)
#                 continue

#             n_total = len(vals)
#             n_exact_zero = int(np.sum(vals == 0.0))
#             nonzero = vals[vals != 0.0]
#             abs_nz = np.abs(nonzero)

#             n_bins = min(80, max(30, n_total // 50))
#             ax.hist(vals, bins=n_bins, color='#4A7FB5', edgecolor='#2A5F95',
#                     linewidth=0.3, alpha=0.85, zorder=2)
#             ax.set_yscale('log')
#             ax.axvline(0, color='#C02020', linewidth=1.0, linestyle='--',
#                        alpha=0.7, zorder=3)

#             ann_parts = [f'N={n_total}']
#             if n_exact_zero > 0:
#                 ann_parts.append(f'zero={n_exact_zero} ({100*n_exact_zero/n_total:.1f}%)')
#             if len(abs_nz) > 0:
#                 ann_parts.append(f'min|v|={abs_nz.min():.2e}')
#                 ann_parts.append(f'P1={np.percentile(vals, 1):.2f}')
#                 ann_parts.append(f'P5={np.percentile(vals, 5):.2f}')

#             ax.text(0.97, 0.97, '\n'.join(ann_parts),
#                     transform=ax.transAxes, fontsize=7.5,
#                     va='top', ha='right',
#                     bbox=dict(boxstyle='round,pad=0.2',
#                               facecolor='white', alpha=0.85, edgecolor='#CCCCCC'))

#             style_axes(ax)

#             if row_i == 0:
#                 ax.set_title(ZONE_SHORT[zone], fontsize=13, fontweight='semibold', pad=6)
#             if col_j == 0:
#                 ax.set_ylabel('Count', fontsize=11)
#                 ax.text(-0.35, 0.5, model, transform=ax.transAxes,
#                         fontsize=12, fontweight='bold', color='#222222',
#                         ha='right', va='center', rotation=90)
#             if row_i == n_rows - 1:
#                 ax.set_xlabel(var_label, fontsize=11)

#     fig_path = DIAG_DIR / f'dist_{var}.png'
#     fig.savefig(fig_path, dpi=160, bbox_inches='tight')
#     print(f'Saved: {fig_path.name}')
#     plt.show()

# print(f'\nDone. Distribution figures saved to {DIAG_DIR}')

## Near-Zero Diagnostic — log₁₀|v| histogram for key variables

Zooms into zero-neighbourhood structure that the full-range histograms cannot resolve.

For each of **Q, prsn, mrros, tran, lai, ET**, a 5×6 (models × zones) panel:
- x-axis: log₁₀|v| of **non-zero** values — reveals gap / spike / smooth tail / quantisation
- y-axis: count (log scale)
- Three vertical ε thresholds at 0.1%, 0.5%, 1% of S (where S = P95−P05 on all_land per model)
- Annotation: exact-zero %, and near-zero % at each ε (fraction with |v| ≤ εS)
- For variables with negative values (Q, ET): separate positive/negative histograms

In [7]:
# # --- Near-zero diagnostic: log10|v| histogram for 6 key variables ---

# NEARZERO_DIR = OUTPUT_DIR / 'diag_nearzero'
# NEARZERO_DIR.mkdir(parents=True, exist_ok=True)

# KEY_VARS = ['Q', 'prsn', 'mrros', 'tran', 'lai', 'ET']
# EPS_FRACS = [0.001, 0.005, 0.01]
# EPS_LABELS = ['0.1%S', '0.5%S', '1%S']
# EPS_COLORS = ['#E63946', '#457B9D', '#2A9D8F']

# def compute_scale(run, var):
#     """S = P95 - P05 on all_land for this model × variable."""
#     vals = get_var_values(run, var, 'all_land')
#     if vals is None or len(vals) < 10:
#         return None
#     return float(np.percentile(vals, 95) - np.percentile(vals, 5))

# COL_W, ROW_H = 3.2, 2.8
# n_rows, n_cols = len(MODELS), len(ZONES)

# for var in KEY_VARS:
#     var_label = UNIT_LABELS.get(var, var)
#     var_full = VAR_FULL_NAMES.get(var, var)

#     scales = {}
#     for run in runs:
#         s = compute_scale(run, var)
#         scales[run['model']] = s

#     fig, axes = plt.subplots(
#         n_rows, n_cols,
#         figsize=(COL_W * n_cols, ROW_H * n_rows),
#         constrained_layout=True, squeeze=False,
#     )
#     fig.suptitle(
#         f'Near-Zero Diagnostic — {var_full} ({var})  |  log₁₀|v|',
#         fontsize=17, fontweight='semibold',
#     )

#     for row_i, run in enumerate(runs):
#         model = run['model']
#         S = scales[model]
#         for col_j, zone in enumerate(ZONES):
#             ax = axes[row_i, col_j]
#             vals = get_var_values(run, var, zone)

#             if vals is None or len(vals) < 5 or S is None or S <= 0:
#                 ax.text(0.5, 0.5, 'no data', ha='center', va='center',
#                         transform=ax.transAxes, fontsize=10, color='#999')
#                 style_axes(ax)
#                 if row_i == 0:
#                     ax.set_title(ZONE_SHORT[zone], fontsize=13, fontweight='semibold', pad=6)
#                 if col_j == 0:
#                     ax.set_ylabel('Count', fontsize=11)
#                     ax.text(-0.35, 0.5, model, transform=ax.transAxes,
#                             fontsize=12, fontweight='bold', color='#222222',
#                             ha='right', va='center', rotation=90)
#                 if row_i == n_rows - 1:
#                     ax.set_xlabel(f'log₁₀|{var}|', fontsize=11)
#                 continue

#             n_total = len(vals)
#             n_exact_zero = int(np.sum(vals == 0.0))
#             nonzero = vals[vals != 0.0]
#             abs_nz = np.abs(nonzero)

#             if len(abs_nz) < 3:
#                 ax.text(0.5, 0.5, f'all zeros\n({n_exact_zero}/{n_total})',
#                         ha='center', va='center',
#                         transform=ax.transAxes, fontsize=10, color='#C03030')
#                 style_axes(ax)
#                 if row_i == 0:
#                     ax.set_title(ZONE_SHORT[zone], fontsize=13, fontweight='semibold', pad=6)
#                 if col_j == 0:
#                     ax.set_ylabel('Count', fontsize=11)
#                     ax.text(-0.35, 0.5, model, transform=ax.transAxes,
#                             fontsize=12, fontweight='bold', color='#222222',
#                             ha='right', va='center', rotation=90)
#                 if row_i == n_rows - 1:
#                     ax.set_xlabel(f'log₁₀|{var}|', fontsize=11)
#                 continue

#             log_abs = np.log10(abs_nz)
#             n_bins = min(80, max(30, len(log_abs) // 40))
#             ax.hist(log_abs, bins=n_bins, color='#4A7FB5', edgecolor='#2A5F95',
#                     linewidth=0.3, alpha=0.85, zorder=2)
#             ax.set_yscale('log')

#             for ef, el, ec in zip(EPS_FRACS, EPS_LABELS, EPS_COLORS):
#                 eps_val = ef * S
#                 if eps_val > 0:
#                     ax.axvline(np.log10(eps_val), color=ec, linewidth=1.2,
#                                linestyle='--', alpha=0.8, zorder=3)

#             ann = [f'N={n_total}  S={S:.1f}']
#             if n_exact_zero > 0:
#                 ann.append(f'=0: {n_exact_zero} ({100*n_exact_zero/n_total:.1f}%)')

#             abs_all = np.abs(vals)
#             for ef, el in zip(EPS_FRACS, EPS_LABELS):
#                 eps_val = ef * S
#                 n_near = int(np.sum(abs_all <= eps_val))
#                 ann.append(f'|v|≤{el}: {n_near} ({100*n_near/n_total:.1f}%)')

#             ax.text(0.97, 0.97, '\n'.join(ann),
#                     transform=ax.transAxes, fontsize=6.5,
#                     va='top', ha='right', family='monospace',
#                     bbox=dict(boxstyle='round,pad=0.2',
#                               facecolor='white', alpha=0.88, edgecolor='#CCCCCC'))

#             style_axes(ax)

#             if row_i == 0:
#                 ax.set_title(ZONE_SHORT[zone], fontsize=13, fontweight='semibold', pad=6)
#             if col_j == 0:
#                 ax.set_ylabel('Count', fontsize=11)
#                 ax.text(-0.35, 0.5, model, transform=ax.transAxes,
#                         fontsize=12, fontweight='bold', color='#222222',
#                         ha='right', va='center', rotation=90)
#             if row_i == n_rows - 1:
#                 ax.set_xlabel(f'log₁₀|{var}|', fontsize=11)

#     from matplotlib.lines import Line2D
#     eps_handles = [
#         Line2D([0], [0], color=ec, linewidth=1.2, linestyle='--', label=el)
#         for el, ec in zip(EPS_LABELS, EPS_COLORS)
#     ]
#     fig.legend(handles=eps_handles, loc='lower center', ncol=3,
#                fontsize=11, frameon=True, fancybox=True,
#                framealpha=0.5, edgecolor='none',
#                bbox_to_anchor=(0.5, -0.03))

#     fig_path = NEARZERO_DIR / f'nearzero_{var}.png'
#     fig.savefig(fig_path, dpi=160, bbox_inches='tight')
#     print(f'Saved: {fig_path.name}')
#     plt.show()

# print(f'\nDone. Near-zero diagnostics saved to {NEARZERO_DIR}')

## Classification — two subsets, same classifier

Both blocks use `classifier0825-branch.py` on RAW points (no trimming).
Saved CSVs include `classifier_sha256` (SHA-256 of that file). Reuse prints a warning if the hash does not match the current classifier.

Decision tree:
1. **Gate 1** — MIC/dCor association screening → else No Global Relationship
2. **Gate 2** — Branch detection → Branch / Candidate / Two-band
3. **Gate 3** — `|Pearson| ≥ 0.5` try Simple (power-law Linear / Near-linear / Acc / Sat);
   if that fails → **No simple**. `|Pearson| < 0.5` → **Complex**. SiZer is not used.

| Block | Points used | CSV | Used by S4.2 |
|---|---|---|---|
| **A** | all finite scatter points | `classification_all_points.csv` | no |
| **B** | drop exact 0 and near-0 (`|v| > 1e-4` on both axes) | `classification_continuous.csv` | yes |

Set `FORCE_RECLASSIFY = True` in the helper cell if you need to recompute an existing CSV.


In [8]:
# --- Shared classifier helper ---

import hashlib
import importlib
import sys

if str(CASE_DIR) not in sys.path:
    sys.path.insert(0, str(CASE_DIR))

_clf_mod = importlib.import_module('classifier0825-branch')
classify_relationship = _clf_mod.classify_relationship

EPS_EFF_CLF = 1e-4
MIN_N_CLF = 30
FORCE_RECLASSIFY = False  # True: ignore existing CSVs and recompute


def classifier_sha256(path=CLASSIFIER_PATH):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


CLASSIFIER_HASH = classifier_sha256()


def stamp_classifier(df):
    out = df.copy()
    out['classifier_file'] = CLASSIFIER_PATH.name
    out['classifier_sha256'] = CLASSIFIER_HASH
    return out


def mask_all_points(x, y):
    return np.asarray(x, float), np.asarray(y, float)


def mask_drop_zero_nearzero(x, y, eps=EPS_EFF_CLF):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = (np.abs(x) > eps) & (np.abs(y) > eps)
    return x[m], y[m]


def classify_all_panels(mask_fn, min_n=MIN_N_CLF, progress_every=50):
    """Run classifier on every pair × model × zone after mask_fn(x, y)."""
    records = []
    total = len(VARIABLE_PAIRS) * len(MODELS) * len(ZONES)
    done = 0
    for x_var, y_var, pair_label in VARIABLE_PAIRS:
        for run in runs:
            model = run['model']
            for zone in ZONES:
                done += 1
                x, y = get_xy(run, x_var, y_var, zone)
                if x is None or len(x) < 5:
                    continue

                n_all = len(x)
                xu, yu = mask_fn(x, y)
                n_used = int(len(xu))
                n_cont = int(((np.abs(x) > EPS_EFF_CLF) & (np.abs(y) > EPS_EFF_CLF)).sum())

                if n_used < min_n:
                    records.append({
                        'Pair': pair_label, 'Model': model, 'Zone': zone,
                        'N_all': n_all, 'N_used': n_used, 'N_cont': n_cont,
                        'group': 'Skipped (too few points)',
                        'gate1_score': None, 'gate1_metric': None,
                        'branch_structure': None, 'pearson_r': None,
                        'power_b': None, 'power_r2': None,
                        'linearity_score': None, 'tp_number': None,
                        'path_trace': 'skipped',
                    })
                    continue

                if done % progress_every == 0:
                    print(f'  [{done}/{total}] {pair_label} / {model} / {zone} (N_used={n_used})')

                result = classify_relationship(xu, yu)
                records.append({
                    'Pair': pair_label, 'Model': model, 'Zone': zone,
                    'N_all': n_all, 'N_used': n_used, 'N_cont': n_cont,
                    'group': result['group'],
                    'gate1_score': result.get('gate1_score'),
                    'gate1_metric': result.get('gate1_metric'),
                    'branch_structure': result.get('branch_structure'),
                    'pearson_r': result.get('pearson_r'),
                    'power_b': result.get('power_b'),
                    'power_r2': result.get('power_r2'),
                    'linearity_score': result.get('linearity_score'),
                    'tp_number': result.get('tp_number'),
                    'path_trace': result.get('path_trace'),
                })
    return stamp_classifier(pd.DataFrame(records))


def summarize_classification(df, title):
    print(f'\n=== {title} ===\n')
    print(f'Panels: {len(df)}')
    if 'classifier_sha256' in df.columns and len(df):
        print(f'classifier_sha256: {df["classifier_sha256"].iloc[0]}')
    print()
    print(df['group'].value_counts().to_string())

    print('\n\n=== Per-pair breakdown ===')
    pair_group = (
        df.groupby('Pair')['group']
        .value_counts()
        .unstack(fill_value=0)
    )
    with pd.option_context('display.max_columns', 20, 'display.width', 160):
        display(pair_group)

    print('\n=== Per-zone breakdown ===')
    display(
        df.groupby('Zone')['group']
        .value_counts()
        .unstack(fill_value=0)
    )

    print('\n=== Per-model breakdown ===')
    display(
        df.groupby('Model')['group']
        .value_counts()
        .unstack(fill_value=0)
    )


def load_or_classify(csv_path, mask_fn, title):
    if csv_path.exists() and not FORCE_RECLASSIFY:
        df = pd.read_csv(csv_path)
        stored = None
        if 'classifier_sha256' in df.columns and len(df):
            stored = str(df['classifier_sha256'].iloc[0])
            if stored in {'', 'nan', 'None'}:
                stored = None
        print(f'Reusing {csv_path.name} ({len(df)} rows). Set FORCE_RECLASSIFY = True to recompute.')
        if stored is None:
            print('WARNING: CSV has no classifier_sha256; cannot verify the classifier version.')
        elif stored != CLASSIFIER_HASH:
            print('WARNING: classifier_sha256 mismatch.')
            print(f'  CSV:     {stored}')
            print(f'  current: {CLASSIFIER_HASH}')
            print('  Set FORCE_RECLASSIFY = True to recompute.')
        else:
            print(f'classifier_sha256 matches ({CLASSIFIER_HASH[:16]}…).')
        return df
    print(f'Classifying: {title}')
    df = classify_all_panels(mask_fn)
    df.to_csv(csv_path, index=False)
    print(f'Saved {csv_path}')
    print(f'classifier_sha256: {CLASSIFIER_HASH}')
    return df

print('Classifier helper ready.')
print(f'Classifier:        {CLASSIFIER_PATH.name}')
print(f'classifier_sha256: {CLASSIFIER_HASH}')
print(f'FORCE_RECLASSIFY = {FORCE_RECLASSIFY}')


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


Classifier helper ready.
FORCE_RECLASSIFY = False


## A — all scatter points

Every finite land cell in the panel, **including** exact zeros (`v = 0`) and near-zeros (`0 < |v| ≤ 1e-4`).

This is what the raw scatter looks like. Ice / dry zeros can still produce Branch here.

Writes `output/S4.1/classification_all_points.csv`.


In [9]:
# --- A: classify all scatter points (zeros kept) ---

ALL_CSV = OUTPUT_DIR / 'classification_all_points.csv'
df_clf_all = load_or_classify(
    ALL_CSV, mask_all_points, 'all scatter points (zeros kept)',
)
summarize_classification(df_clf_all, 'A — all scatter points')


Classifying: all scatter points (zeros kept)
  [50/510] ET → Q / GFDL-CM4 / WW (N_used=2653)
  [100/510] prsn → Q / CNRM-CM6-1 / CW (N_used=2821)
  [150/510] tran → Q / CMCC-CM2-SR5 / LI (N_used=6967)
  [200/510] hfls → Q / GFDL-CM4 / WW (N_used=2653)
  [250/510] lai → Q / CNRM-CM6-1 / CW (N_used=2821)
  [300/510] tas → Q / CMCC-CM2-SR5 / LI (N_used=6967)
  [350/510] mrso → Q / GFDL-CM4 / WW (N_used=2653)
  [400/510] rlds → Q / CNRM-CM6-1 / CW (N_used=2821)
  [450/510] rlus → Q / CMCC-CM2-SR5 / LI (N_used=6967)
  [500/510] P → ET / GFDL-CM4 / WW (N_used=2653)
Saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4.1/classification_all_points.csv

=== A — all scatter points ===

Panels: 510

group
Complex                   253
No Global Relationship     90
No simple                  80
Near-linear                27
Acceleration               24
Linear                     22
Saturation                  8
Branch                      3
Candidate branch       

group,Acceleration,Branch,Candidate branch,Complex,Linear,Near-linear,No Global Relationship,No simple,Saturation
Pair,,,,,,,,,
ET → Q,0,0,0,17,0,0,8,5,0
P → ET,0,0,0,6,1,7,0,10,6
P → Q,15,2,1,0,1,11,0,0,0
evspsblsoi → Q,0,0,0,18,0,0,9,3,0
hfls → Q,0,0,0,17,0,0,8,5,0
hfss → Q,0,0,0,22,0,0,7,1,0
lai → Q,0,0,0,10,0,0,7,13,0
mrros → Q,1,0,0,1,18,9,0,0,1
mrso → Q,0,0,0,20,0,0,6,4,0



=== Per-zone breakdown ===


group,Acceleration,Branch,Candidate branch,Complex,Linear,Near-linear,No Global Relationship,No simple,Saturation
Zone,,,,,,,,,
CD,5,0,0,52,3,4,10,11,0
CW,2,0,0,29,2,5,37,8,2
LI,7,2,3,26,7,4,4,31,1
WD,6,0,0,42,2,2,11,17,5
WW,1,0,0,47,4,5,21,7,0
all_land,3,1,0,57,4,7,7,6,0



=== Per-model breakdown ===


group,Acceleration,Branch,Candidate branch,Complex,Linear,Near-linear,No Global Relationship,No simple,Saturation
Model,,,,,,,,,
CESM2,3,2,0,49,6,6,15,19,2
CMCC-CM2-SR5,6,0,1,48,2,6,21,17,1
CNRM-CM6-1,3,0,0,39,7,6,33,14,0
CanESM5,6,0,0,66,2,6,7,12,3
GFDL-CM4,6,1,2,51,5,3,14,18,2


## B — drop 0 and near-0

Keep only continuous points: `|x| > 1e-4` AND `|y| > 1e-4`.

This is the official classification for the S4.2 figures.

Writes `output/S4.1/classification_continuous.csv`. If that file already exists, this cell reuses it unless you set `FORCE_RECLASSIFY = True` in the helper cell and re-run.


In [10]:
# --- B: classify after dropping 0 and near-0 (|v| > 1e-4) ---

CONT_CSV = OUTPUT_DIR / 'classification_continuous.csv'
df_clf_cont = load_or_classify(
    CONT_CSV, mask_drop_zero_nearzero, 'continuous only (|v| > 1e-4)',
)

# S4.2 expects N_cont. New runs also have N_used (= N_cont under this mask).
if 'N_cont' not in df_clf_cont.columns:
    df_clf_cont['N_cont'] = df_clf_cont['N_used']

df_clf = df_clf_cont.copy()  # older figure cells / lookup may use this name

summarize_classification(df_clf_cont, 'B — drop 0 and near-0 (|v| > 1e-4)')
print(f'\nS4.2 reads: {CONT_CSV}')


Reusing classification_continuous.csv (510 rows). Set FORCE_RECLASSIFY = True to recompute.

=== B — drop 0 and near-0 (|v| > 1e-4) ===

Panels: 510

group
Complex                   246
No Global Relationship    103
No simple                  72
Acceleration               27
Near-linear                27
Linear                     23
Saturation                  9
Skipped (too few cont)      2
Candidate branch            1


=== Per-pair breakdown ===


group,Acceleration,Candidate branch,Complex,Linear,Near-linear,No Global Relationship,No simple,Saturation,Skipped (too few cont)
Pair,,,,,,,,,
ET → Q,0,0,15,0,0,10,4,1,0
P → ET,0,0,6,1,7,0,10,6,0
P → Q,16,1,0,2,11,0,0,0,0
evspsblsoi → Q,0,0,18,0,0,11,1,0,0
hfls → Q,0,0,15,0,0,10,4,1,0
hfss → Q,0,0,22,0,0,8,0,0,0
lai → Q,0,0,12,0,0,7,10,0,1
mrros → Q,1,0,1,18,9,0,0,1,0
mrso → Q,0,0,21,0,0,6,3,0,0



=== Per-zone breakdown ===


group,Acceleration,Candidate branch,Complex,Linear,Near-linear,No Global Relationship,No simple,Saturation,Skipped (too few cont)
Zone,,,,,,,,,
CD,5,0,53,3,4,10,10,0,0
CW,2,0,29,2,5,37,8,2,0
LI,9,1,32,8,5,3,23,2,2
WD,6,0,40,2,2,10,20,5,0
WW,1,0,44,4,5,24,7,0,0
all_land,4,0,48,4,6,19,4,0,0



=== Per-model breakdown ===


group,Acceleration,Candidate branch,Complex,Linear,Near-linear,No Global Relationship,No simple,Saturation,Skipped (too few cont)
Model,,,,,,,,,
CESM2,3,0,44,6,6,24,18,1,0
CMCC-CM2-SR5,6,1,50,2,6,22,14,1,0
CNRM-CM6-1,3,0,37,7,6,33,16,0,0
CanESM5,6,0,64,2,6,5,14,3,2
GFDL-CM4,9,0,51,6,3,19,10,4,0



S4.2 reads: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4.1/classification_continuous.csv


## A vs B — which labels change

Same classifier, different point set (`|v| > 1e-4` cut, not the exact-zero inventory above).

Three different counts:
- **Dropped points:** `N_cont < N_all` — the panel lost some zeros / near-zeros
- **Label unchanged after drop:** points were removed but the class stayed the same
- **Label changed:** the class actually moved (this is the A vs B result that matters)

Do not use the earlier exact-zero inventory as a substitute for these numbers.


In [11]:
# --- Compare A (all points) vs B (drop 0 / near-0) ---

if 'df_clf_all' not in globals():
    raise RuntimeError('Run block A first.')
if 'df_clf_cont' not in globals():
    raise RuntimeError('Run block B first.')


def _take(df, tag):
    out = pd.DataFrame({
        'Pair': df['Pair'].to_numpy(),
        'Model': df['Model'].to_numpy(),
        'Zone': df['Zone'].to_numpy(),
        f'group_{tag}': df['group'].to_numpy(),
    })
    # A keeps N_all; B keeps N_cont. Do not copy both onto both sides
    # or merge will rename them to N_cont_x / N_cont_y.
    if tag == 'all' and 'N_all' in df.columns:
        out['N_all'] = df['N_all'].to_numpy()
    if tag == 'cont':
        if 'N_cont' in df.columns:
            out['N_cont'] = df['N_cont'].to_numpy()
        elif 'N_used' in df.columns:
            out['N_cont'] = df['N_used'].to_numpy()
    if 'branch_structure' in df.columns:
        out[f'branch_{tag}'] = df['branch_structure'].to_numpy()
    if 'pearson_r' in df.columns:
        out[f'pearson_{tag}'] = df['pearson_r'].to_numpy()
    if 'power_b' in df.columns:
        out[f'b_{tag}'] = df['power_b'].to_numpy()
    if 'path_trace' in df.columns:
        out[f'path_{tag}'] = df['path_trace'].to_numpy()
    return out


a = _take(df_clf_all, 'all')
b = _take(df_clf_cont, 'cont')
merged = a.merge(b, on=['Pair', 'Model', 'Zone'], how='inner')


def _canon(s):
    s = str(s)
    if s.startswith('Skipped'):
        return 'Skipped'
    return s


merged['dropped_points'] = merged['N_cont'] < merged['N_all']
merged['changed'] = (
    merged['group_all'].map(_canon) != merged['group_cont'].map(_canon)
)

n_all = len(merged)
n_drop = int(merged['dropped_points'].sum())
n_keep = n_all - n_drop
n_changed = int(merged['changed'].sum())
n_drop_unchanged = int((merged['dropped_points'] & ~merged['changed']).sum())
n_drop_changed = int((merged['dropped_points'] & merged['changed']).sum())
n_keep_changed = int((~merged['dropped_points'] & merged['changed']).sum())

print('=== A vs B counts (|v| > 1e-4 cut) ===')
print('Not the exact-zero inventory above.\n')
print(f'Comparable panels:                         {n_all}')
print(f'Dropped some points (N_cont < N_all):      {n_drop}')
print(f'  label unchanged after drop:              {n_drop_unchanged}')
print(f'  label CHANGED after drop:                {n_drop_changed}')
print(f'No points dropped:                         {n_keep}')
print(f'  of which label still changed:            {n_keep_changed}')
print(f'Total label changes (A vs B):              {n_changed}')

print('\n=== Crosstab  A (rows) × B (columns) ===')
ct = pd.crosstab(merged['group_all'], merged['group_cont'], margins=True)
with pd.option_context('display.max_columns', 20, 'display.width', 180):
    display(ct)

changed = merged.loc[merged['changed']].sort_values(['Pair', 'Model', 'Zone'])
show = [c for c in [
    'Pair', 'Model', 'Zone', 'N_all', 'N_cont',
    'group_all', 'group_cont', 'branch_all', 'branch_cont',
] if c in changed.columns]

print(f'\n=== Panels whose label changed ({len(changed)}) ===')
with pd.option_context('display.max_rows', 400, 'display.width', 180):
    display(changed[show].reset_index(drop=True))

print('\n=== Changed count by pair ===')
if len(changed):
    display(changed.groupby('Pair').size().rename('n_changed').to_frame())
else:
    print('(none)')

print('\n=== Changed count by zone ===')
if len(changed):
    display(changed.groupby('Zone').size().rename('n_changed').to_frame())
else:
    print('(none)')

if 'CLASSIFIER_HASH' in globals():
    merged['classifier_file'] = CLASSIFIER_PATH.name
    merged['classifier_sha256'] = CLASSIFIER_HASH

cmp_path = OUTPUT_DIR / 'classification_A_vs_B.csv'
merged.to_csv(cmp_path, index=False)
print(f'\nComparison table saved to {cmp_path.name}')
print('S4.2 still uses classification_continuous.csv (block B).')


Comparable panels: 510
Unchanged label:   464
Changed label:     46

=== Crosstab  A (rows) × B (columns) ===


group_cont,Acceleration,Candidate branch,Complex,Linear,Near-linear,No Global Relationship,No simple,Saturation,Skipped (too few cont),All
group_all,,,,,,,,,,
Acceleration,23,0,0,0,0,0,1,0,0,24
Branch,0,0,1,1,1,0,0,0,0,3
Candidate branch,1,1,1,0,0,0,0,0,0,3
Complex,2,0,233,0,0,14,4,0,0,253
Linear,0,0,0,22,0,0,0,0,0,22
Near-linear,1,0,0,0,26,0,0,0,0,27
No Global Relationship,0,0,0,0,0,87,1,0,2,90
No simple,0,0,11,0,0,2,65,2,0,80
Saturation,0,0,0,0,0,0,1,7,0,8



=== Panels whose label changed (46) ===


,Pair,Model,Zone,N_all,group_all,group_cont,branch_all,branch_cont
0,ET → Q,CESM2,all_land,21013,Complex,No Global Relationship,No,NaN
1,ET → Q,GFDL-CM4,LI,7273,No simple,Saturation,No,No
2,ET → Q,GFDL-CM4,all_land,22514,Complex,No Global Relationship,No,NaN
3,P → Q,CESM2,LI,7022,Branch,Near-linear,Yes,No
4,P → Q,CESM2,all_land,21013,Near-linear,Acceleration,No,No
5,P → Q,GFDL-CM4,LI,7273,Branch,Linear,Yes,No
6,evspsblsoi → Q,CESM2,LI,7022,No simple,Complex,No,No
7,evspsblsoi → Q,CESM2,all_land,21013,Complex,No Global Relationship,No,NaN
8,evspsblsoi → Q,GFDL-CM4,LI,7273,No simple,Complex,No,No
9,evspsblsoi → Q,GFDL-CM4,all_land,22514,Complex,No Global Relationship,No,NaN



=== Changed count by pair ===


,n_changed
Pair,
ET → Q,3
P → Q,3
evspsblsoi → Q,4
hfls → Q,3
hfss → Q,2
lai → Q,6
mrso → Q,1
mrsos → Q,1
prsn → Q,9



=== Changed count by zone ===


,n_changed
Zone,
CD,1
LI,20
WD,3
WW,3
all_land,19



Comparison table saved to classification_A_vs_B.csv
S4.2 still uses classification_continuous.csv (block B).


## Figures moved to S4.2

The official 5×6 scatter / hexbin figures — class-coloured panels, LOWESS (magenta) and GAM (blue dashed) on **both** scatter and hexbin — are in:

`S4.2_CMIP6_continuous_scatter_fits.ipynb`

It reads `output/S4.1/classification_continuous.csv`. You do not need to re-run classification.

The original near-zero 5×6 and symlog cells below are disabled (`%%script false`).


In [12]:
%%script --no-raise-error false
# Disabled: figures moved to S4.2_CMIP6_continuous_scatter_fits.ipynb
# --- Near-zero scatter diagnostic: three-level zero structure ---

from IPython.display import display

NZ_SCATTER_DIR = OUTPUT_DIR / 'nearzero_scatter'
NZ_SCATTER_DIR.mkdir(parents=True, exist_ok=True)

EPS_TINY = 1e-6
EPS_EFF  = 1e-4


def zero_groups_eps(x, y, eps_tiny=EPS_TINY, eps_eff=EPS_EFF):
    """Split into 4 groups: continuous, effective-zero, tiny numerical, exact-zero."""
    xz = x == 0.0
    yz = y == 0.0
    exact_zero = xz | yz

    x_tiny = (~xz) & (np.abs(x) <= eps_tiny)
    y_tiny = (~yz) & (np.abs(y) <= eps_tiny)
    x_eff = (~xz) & (np.abs(x) > eps_tiny) & (np.abs(x) <= eps_eff)
    y_eff = (~yz) & (np.abs(y) > eps_tiny) & (np.abs(y) <= eps_eff)

    near_zero = (~exact_zero) & (x_tiny | y_tiny)
    eff_zero = (~exact_zero) & (~near_zero) & (x_eff | y_eff)
    cont = (~exact_zero) & (~near_zero) & (~eff_zero)

    return {
        'cont':       cont,
        'eff_zero':   eff_zero,
        'near_zero':  near_zero,
        'exact_zero': exact_zero,
    }


NZ_COLORS = {
    'cont':       '#444444',
    'eff_zero':   '#7CB342',
    'near_zero':  '#FF8C00',
    'exact_zero': '#7020A0',
}
NZ_SIZES = {
    'cont':       1.0,
    'eff_zero':   3.5,
    'near_zero':  4.0,
    'exact_zero': 4.0,
}
NZ_ALPHAS = {
    'cont':       0.5,
    'eff_zero':   0.85,
    'near_zero':  0.9,
    'exact_zero': 0.9,
}
NZ_DRAW_ORDER = ['cont', 'eff_zero', 'near_zero', 'exact_zero']
NZ_LEGEND_LABELS = {
    'cont':       f'Continuous (|v| > {EPS_EFF:.0e})',
    'eff_zero':   f'Effective zero ({EPS_TINY:.0e} < |v| ≤ {EPS_EFF:.0e})',
    'near_zero':  f'Tiny numerical (0 < |v| ≤ {EPS_TINY:.0e})',
    'exact_zero': 'Exact zero (v = 0)',
}

COL_W, ROW_H = 3.2, 2.8
n_rows, n_cols = len(MODELS), len(ZONES)
fig_count = 0

for x_var, y_var, pair_label in VARIABLE_PAIRS:
    pair_slug = f'{x_var}_{y_var}'
    x_label = UNIT_LABELS.get(x_var, x_var)
    y_label = UNIT_LABELS.get(y_var, y_var)
    pair_full = full_pair_title(x_var, y_var)

    col_limits = {}
    for zone in ZONES:
        col_limits[zone] = column_limits_for_pair(x_var, y_var, zone, runs)
    cls_lookup = classification_lookup()

    # ============================================================
    # Figure 1: Scatter — three-level zero structure
    # ============================================================
    fig_sc, axes_sc = plt.subplots(
        n_rows, n_cols,
        figsize=(COL_W * n_cols, ROW_H * n_rows),
        constrained_layout=True, squeeze=False,
    )
    fig_sc.suptitle(
        f'Scatter (zero structure) — {pair_full}',
        fontsize=17, fontweight='semibold',
    )

    for row_i, run in enumerate(runs):
        model = run['model']
        for col_j, zone in enumerate(ZONES):
            ax = axes_sc[row_i, col_j]
            ax.set_aspect('auto')
            x, y = get_xy(run, x_var, y_var, zone)
            xlim, ylim = col_limits[zone]
            grp = cls_lookup.get((pair_label, model, zone))
            ax.set_facecolor(class_face(grp))

            if x is None or len(x) < 5:
                ax.text(0.5, 0.5, 'no data', ha='center', va='center',
                        transform=ax.transAxes, fontsize=10, color='#999')
                ax.set_xlim(xlim); ax.set_ylim(ylim)
                style_axes(ax)
                if row_i == 0:
                    ax.set_title(ZONE_SHORT[zone], fontsize=13, fontweight='semibold', pad=6)
                if col_j == 0:
                    ax.set_ylabel(y_label, fontsize=11)
                    ax.text(-0.35, 0.5, model, transform=ax.transAxes,
                            fontsize=12, fontweight='bold', color='#222222',
                            ha='right', va='center', rotation=90)
                if row_i == n_rows - 1:
                    ax.set_xlabel(x_label, fontsize=11)
                continue

            g = zero_groups_eps(x, y)
            n_all = len(x)
            n_cont = int(g['cont'].sum())
            n_eff = int(g['eff_zero'].sum())
            n_nz = int(g['near_zero'].sum())
            n_ez = int(g['exact_zero'].sum())

            for key in NZ_DRAW_ORDER:
                m = g[key]
                if m.sum() == 0:
                    continue
                ax.scatter(x[m], y[m], c=NZ_COLORS[key],
                           s=NZ_SIZES[key], alpha=NZ_ALPHAS[key],
                           linewidths=0, rasterized=True,
                           zorder=2 if key == 'cont' else 3)

            ann_parts = [f'N={n_all}  cont={n_cont}']
            if n_eff > 0:
                ann_parts.append(f'eff0={n_eff} ({100*n_eff/n_all:.1f}%)')
            if n_nz > 0:
                ann_parts.append(f'tiny={n_nz} ({100*n_nz/n_all:.1f}%)')
            if n_ez > 0:
                ann_parts.append(f'ez={n_ez} ({100*n_ez/n_all:.1f}%)')
            draw_panel_class_label(ax, grp, ann_parts, fontsize=7.5)

            ax.set_xlim(xlim); ax.set_ylim(ylim)
            style_axes(ax)

            if row_i == 0:
                ax.set_title(ZONE_SHORT[zone], fontsize=13, fontweight='semibold', pad=6)
            if col_j == 0:
                ax.set_ylabel(y_label, fontsize=11)
                ax.text(-0.35, 0.5, model, transform=ax.transAxes,
                        fontsize=12, fontweight='bold', color='#222222',
                        ha='right', va='center', rotation=90)
            if row_i == n_rows - 1:
                ax.set_xlabel(x_label, fontsize=11)

    handles = [
        mpatches.Patch(color=NZ_COLORS[k], label=NZ_LEGEND_LABELS[k])
        for k in NZ_DRAW_ORDER
    ]
    fig_sc.legend(handles=handles, loc='lower center', ncol=4,
                  fontsize=10, frameon=True, fancybox=True,
                  framealpha=0.5, edgecolor='none',
                  bbox_to_anchor=(0.5, -0.04))

    sc_path = NZ_SCATTER_DIR / f'nz_scatter_{pair_slug}.png'
    fig_sc.savefig(sc_path, dpi=160, bbox_inches='tight')
    print(f'Saved: {sc_path.name}')
    display(fig_sc)
    plt.close(fig_sc)
    fig_count += 1

    # ============================================================
    # Figure 2: Hexbin — Continuous only (|x| > 1e-4 AND |y| > 1e-4)
    # ============================================================
    fig_hc, axes_hc = plt.subplots(
        n_rows, n_cols,
        figsize=(COL_W * n_cols, ROW_H * n_rows),
        constrained_layout=True, squeeze=False,
    )
    fig_hc.suptitle(
        f'Hexbin Density — Continuous (|v| > {EPS_EFF:.0e}) — {pair_full}',
        fontsize=17, fontweight='semibold',
    )

    for row_i, run in enumerate(runs):
        model = run['model']
        for col_j, zone in enumerate(ZONES):
            ax = axes_hc[row_i, col_j]
            ax.set_aspect('auto')
            x, y = get_xy(run, x_var, y_var, zone)
            xlim, ylim = col_limits[zone]
            grp = cls_lookup.get((pair_label, model, zone))
            ax.set_facecolor(class_face(grp))

            if x is None or len(x) < 5:
                ax.text(0.5, 0.5, 'no data', ha='center', va='center',
                        transform=ax.transAxes, fontsize=10, color='#999')
                ax.set_xlim(xlim); ax.set_ylim(ylim)
                style_axes(ax)
                if row_i == 0:
                    ax.set_title(ZONE_SHORT[zone], fontsize=13, fontweight='semibold', pad=6)
                if col_j == 0:
                    ax.set_ylabel(y_label, fontsize=11)
                    ax.text(-0.35, 0.5, model, transform=ax.transAxes,
                            fontsize=12, fontweight='bold', color='#222222',
                            ha='right', va='center', rotation=90)
                if row_i == n_rows - 1:
                    ax.set_xlabel(x_label, fontsize=11)
                continue

            g = zero_groups_eps(x, y)
            n_all = len(x)
            n_cont = int(g['cont'].sum())
            ret = 100 * n_cont / n_all if n_all > 0 else 0
            xc, yc = x[g['cont']], y[g['cont']]

            if n_cont < 20:
                ax.text(0.5, 0.5,
                        f'Too few continuous\nN_cont={n_cont}',
                        ha='center', va='center',
                        transform=ax.transAxes, fontsize=10, color='#C03030')
                ax.set_xlim(xlim); ax.set_ylim(ylim)
                style_axes(ax)
                if row_i == 0:
                    ax.set_title(ZONE_SHORT[zone], fontsize=13, fontweight='semibold', pad=6)
                if col_j == 0:
                    ax.set_ylabel(y_label, fontsize=11)
                    ax.text(-0.35, 0.5, model, transform=ax.transAxes,
                            fontsize=12, fontweight='bold', color='#222222',
                            ha='right', va='center', rotation=90)
                if row_i == n_rows - 1:
                    ax.set_xlabel(x_label, fontsize=11)
                continue

            draw_density_hexbin(
                ax, xc, yc, cmap='Greys', gridsize=26,
                extent=(xlim[0], xlim[1], ylim[0], ylim[1]),
            )

            n_eff = int(g['eff_zero'].sum())
            n_nz = int(g['near_zero'].sum())
            n_ez = int(g['exact_zero'].sum())
            draw_panel_class_label(
                ax, grp,
                [f'cont={n_cont} ({ret:.0f}%)',
                 f'eff0={n_eff}  tiny={n_nz}  ez={n_ez}'],
                fontsize=7.5,
            )

            ax.set_xlim(xlim); ax.set_ylim(ylim)
            style_axes(ax)

            if row_i == 0:
                ax.set_title(ZONE_SHORT[zone], fontsize=13, fontweight='semibold', pad=6)
            if col_j == 0:
                ax.set_ylabel(y_label, fontsize=11)
                ax.text(-0.35, 0.5, model, transform=ax.transAxes,
                        fontsize=12, fontweight='bold', color='#222222',
                        ha='right', va='center', rotation=90)
            if row_i == n_rows - 1:
                ax.set_xlabel(x_label, fontsize=11)

    hc_path = NZ_SCATTER_DIR / f'nz_hexbin_{pair_slug}.png'
    fig_hc.savefig(hc_path, dpi=160, bbox_inches='tight')
    print(f'Saved: {hc_path.name}')
    display(fig_hc)
    plt.close(fig_hc)
    fig_count += 1

print(f'\nDone. {fig_count} figures saved to {NZ_SCATTER_DIR}')


## Symlog scatter — disabled

Unused for the S4.2 figure set. The cell below is disabled.


In [13]:
%%script --no-raise-error false
# Disabled: figures moved to S4.2_CMIP6_continuous_scatter_fits.ipynb
# --- Symlog scatter: zoom into near-zero region ---
# Axes limited to [-10, 10] with linthresh=1e-5 so that
# 1e-4, 1e-2 are evenly spaced in log region.

SYMLOG_DIR = OUTPUT_DIR / 'symlog_scatter'
SYMLOG_DIR.mkdir(parents=True, exist_ok=True)

ZERO_VARS = {'Q', 'prsn', 'mrros', 'tran', 'lai', 'ET', 'evspsblsoi'}
SYMLOG_LINTHRESH = 1e-5
SYMLOG_RANGE = (-10, 10)


def _symlog_label(val, pos):
    if val == 0:
        return '0'
    exp = np.log10(abs(val))
    if abs(exp - round(exp)) < 0.01:
        e = int(round(exp))
        sign = '−' if val < 0 else ''
        if e == 0:
            return f'{sign}1'
        return f'{sign}1e{e}'
    return f'{val:.0e}'

_symlog_fmt = mticker.FuncFormatter(_symlog_label)


def set_symlog_ticks(ax, axis, lo, hi):
    """Place ticks at 0, +/-1e-4, +/-1e-2, +/-1."""
    pos_ticks = [1e-4, 1e-2, 1]
    ticks = [0]
    ticks += [t for t in pos_ticks if t <= hi]
    ticks += [-t for t in pos_ticks if -t >= lo]
    ticks = sorted(set(ticks))
    if axis == 'x':
        ax.set_xticks(ticks)
        ax.xaxis.set_major_formatter(_symlog_fmt)
        ax.xaxis.set_minor_locator(mticker.NullLocator())
    else:
        ax.set_yticks(ticks)
        ax.yaxis.set_major_formatter(_symlog_fmt)
        ax.yaxis.set_minor_locator(mticker.NullLocator())


COL_W, ROW_H = 3.2, 2.8
n_rows, n_cols = len(MODELS), len(ZONES)
fig_count = 0

for x_var, y_var, pair_label in VARIABLE_PAIRS:
    pair_slug = f'{x_var}_{y_var}'
    x_label = UNIT_LABELS.get(x_var, x_var)
    y_label = UNIT_LABELS.get(y_var, y_var)
    pair_full = full_pair_title(x_var, y_var)

    use_symlog_x = x_var in ZERO_VARS
    use_symlog_y = y_var in ZERO_VARS

    col_limits = {}
    for zone in ZONES:
        col_limits[zone] = column_limits_for_pair(x_var, y_var, zone, runs)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(COL_W * n_cols, ROW_H * n_rows),
        constrained_layout=True, squeeze=False,
    )

    scale_note = []
    if use_symlog_x:
        scale_note.append(f'x: symlog(lt={SYMLOG_LINTHRESH:.0e})')
    if use_symlog_y:
        scale_note.append(f'y: symlog(lt={SYMLOG_LINTHRESH:.0e})')
    scale_str = '  |  '.join(scale_note) if scale_note else 'linear'

    fig.suptitle(
        f'Scatter (symlog) — {pair_full}\n{scale_str}',
        fontsize=15, fontweight='semibold',
    )

    for row_i, run in enumerate(runs):
        model = run['model']
        for col_j, zone in enumerate(ZONES):
            ax = axes[row_i, col_j]
            ax.set_aspect('auto')
            x, y = get_xy(run, x_var, y_var, zone)
            xlim_full, ylim_full = col_limits[zone]

            if x is None or len(x) < 5:
                ax.text(0.5, 0.5, 'no data', ha='center', va='center',
                        transform=ax.transAxes, fontsize=10, color='#999')
                style_axes(ax)
                if row_i == 0:
                    ax.set_title(ZONE_SHORT[zone], fontsize=13, fontweight='semibold', pad=6)
                if col_j == 0:
                    ax.set_ylabel(y_label, fontsize=11)
                    ax.text(-0.35, 0.5, model, transform=ax.transAxes,
                            fontsize=12, fontweight='bold', color='#222222',
                            ha='right', va='center', rotation=90)
                if row_i == n_rows - 1:
                    ax.set_xlabel(x_label, fontsize=11)
                continue

            g = zero_groups_eps(x, y)
            n_all = len(x)
            n_cont = int(g['cont'].sum())
            n_eff = int(g['eff_zero'].sum())
            n_nz = int(g['near_zero'].sum())
            n_ez = int(g['exact_zero'].sum())

            for key in NZ_DRAW_ORDER:
                m = g[key]
                if m.sum() == 0:
                    continue
                ax.scatter(x[m], y[m], c=NZ_COLORS[key],
                           s=NZ_SIZES[key], alpha=NZ_ALPHAS[key],
                           linewidths=0, rasterized=True,
                           zorder=2 if key == 'cont' else 3)

            if use_symlog_x:
                ax.set_xscale('symlog', linthresh=SYMLOG_LINTHRESH)
            if use_symlog_y:
                ax.set_yscale('symlog', linthresh=SYMLOG_LINTHRESH)

            xlim = SYMLOG_RANGE if use_symlog_x else xlim_full
            ylim = SYMLOG_RANGE if use_symlog_y else ylim_full
            ax.set_xlim(xlim)
            ax.set_ylim(ylim)

            if use_symlog_x:
                set_symlog_ticks(ax, 'x', xlim[0], xlim[1])
            if use_symlog_y:
                set_symlog_ticks(ax, 'y', ylim[0], ylim[1])

            ann_parts = [f'N={n_all}  cont={n_cont}']
            if n_eff > 0:
                ann_parts.append(f'eff0={n_eff} ({100*n_eff/n_all:.1f}%)')
            if n_nz > 0:
                ann_parts.append(f'tiny={n_nz} ({100*n_nz/n_all:.1f}%)')
            if n_ez > 0:
                ann_parts.append(f'ez={n_ez} ({100*n_ez/n_all:.1f}%)')
            ax.text(0.03, 0.97, '\n'.join(ann_parts),
                    transform=ax.transAxes, fontsize=7.5,
                    va='top', ha='left',
                    bbox=dict(boxstyle='round,pad=0.2',
                              facecolor='white', alpha=0.85, edgecolor='#CCCCCC'))

            style_axes(ax)
            ax.tick_params(axis='both', labelsize=7)

            if row_i == 0:
                ax.set_title(ZONE_SHORT[zone], fontsize=13, fontweight='semibold', pad=6)
            if col_j == 0:
                ax.set_ylabel(y_label, fontsize=11)
                ax.text(-0.35, 0.5, model, transform=ax.transAxes,
                        fontsize=12, fontweight='bold', color='#222222',
                        ha='right', va='center', rotation=90)
            if row_i == n_rows - 1:
                ax.set_xlabel(x_label, fontsize=11)

    handles = [
        mpatches.Patch(color=NZ_COLORS[k], label=NZ_LEGEND_LABELS[k])
        for k in NZ_DRAW_ORDER
    ]
    fig.legend(handles=handles, loc='lower center', ncol=4,
               fontsize=10, frameon=True, fancybox=True,
               framealpha=0.5, edgecolor='none',
               bbox_to_anchor=(0.5, -0.04))

    fig_path = SYMLOG_DIR / f'symlog_{pair_slug}.png'
    fig.savefig(fig_path, dpi=160, bbox_inches='tight')
    print(f'Saved: {fig_path.name}')
    plt.show()
    fig_count += 1

print(f'\nDone. {fig_count} figures saved to {SYMLOG_DIR}')
